# Analyze JobA

Sections:
1. **Setup** - imports, paths, model list.
2. **Loader** - single function that walks an output directory and returns a `DataFrame` keyed by `(category, model)`.
3. **Section A - JobA aggregate (all loaded val_f1 models)**: per-cell table, per-model means/medians, per-cat winners + win counts, top-10 AUPR, hardest/easiest cats, per-defect recall, industrial metrics.
4. **Section B - Clean <-> val_defect comparison (all paired models)**: Tables A-D, sanity gates, threshold-shift summary.
5. **Section C - Plots**: F1-lift bar chart, AUROC stability scatter, threshold-ratio histogram.

After the run, Tables A-D are also persisted as TSV under `data/outputs/jobA_val_defect_V1/_analysis/` (drop-in replacement for `compare_clean_vd.py`'s output).

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path
from statistics import mean, median, stdev

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent

OUTPUTS = REPO / "data" / "outputs"
JOBA_CLEAN_DIR = OUTPUTS / "JobA_v0"
JOBA_VD_DIR = OUTPUTS / "jobA_val_defect_V1"
ANALYSIS_DIR = JOBA_VD_DIR / "_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_MODELS = ["anomalib_patchcore", "anomalib_padim", "subspacead"]
TRAINED_MODELS = ["anomalib_stfpm", "anomalib_csflow", "anomalib_draem", "rd4ad"]
ALL_MODELS = FEATURE_MODELS + TRAINED_MODELS

LEGACY_METRICS = [
    "auroc", "aupr", "f1", "precision", "recall", "accuracy",
    "ms_per_image", "fps", "peak_vram_mb", "fit_seconds", "predict_seconds",
    "threshold_used", "threshold_mode",
    "train_samples", "val_samples", "test_samples",
    "val_precision", "val_recall", "val_f1", "val_auroc", "val_aupr",
]
INDUSTRIAL_METRICS = [
    "recall_at_fpr_1pct", "recall_at_fpr_5pct",
    "macro_recall", "weighted_recall",
]
ALL_METRICS = LEGACY_METRICS + INDUSTRIAL_METRICS

print(f"Repo:         {REPO}")
print(f"Clean baseline (JobA_v0):     {JOBA_CLEAN_DIR.exists()} -> {JOBA_CLEAN_DIR}")
print(f"val_defect rerun (V1):        {JOBA_VD_DIR.exists()} -> {JOBA_VD_DIR}")

## 2. Loader

Walks a directory whose subfolders look like `<prefix><category>_<UTC>/benchmark_summary.json`. The category is the substring between the prefix and the timestamp suffix. Returns a tidy `DataFrame` with one row per `(category, model)` cell.

Notes:
- `csflow` / `stfpm` / `draem` / `rd4ad` are *trained* models that ran model-major (one model at a time per category), so they are loaded through their own prefixes and then merged back into the unified comparison tables.
- When the same `(cat, model)` pair has multiple timestamps, the freshest one wins.

In [ ]:
_TS_TAIL = r"_\d{8}_\d{6}$"


def _category_from_dirname(dirname: str, prefix: str) -> str | None:
    pattern = rf"^{re.escape(prefix)}(.+){_TS_TAIL}"
    m = re.match(pattern, dirname)
    return m.group(1) if m else None


def load_runs(
    directory: Path,
    prefix: str,
    models: list[str] = FEATURE_MODELS,
    skip_substrings: tuple[str, ...] = (),
) -> pd.DataFrame:
    """Walk `directory` and return one row per (category, model) freshest run.

    `prefix` is the directory-name prefix that identifies the run family
    (e.g. ``jobA_``, ``jobA_val_defect_``, ``jobA_anomalib_csflow_``).
    """
    rows: dict[tuple[str, str], dict] = {}
    if not directory.exists():
        return pd.DataFrame()
    for d in sorted(directory.iterdir()):
        if not d.is_dir():
            continue
        name = d.name
        if any(s in name for s in skip_substrings):
            continue
        category = _category_from_dirname(name, prefix)
        if category is None:
            continue
        fp = d / "benchmark_summary.json"
        if not fp.exists():
            continue
        data = json.loads(fp.read_text())
        split = data.get("dataset", {}).get("split", {})
        for entry in data.get("models", []):
            mdl = entry.get("model")
            if mdl not in models:
                continue
            key = (category, mdl)
            prev = rows.get(key)
            if prev is not None and prev["_dir"] >= name:
                continue
            row = {"category": category, "model": mdl, "_dir": name,
                   "run_id": data.get("run", {}).get("run_id")}
            for k in ALL_METRICS:
                row[k] = entry.get(k)
            row["per_defect_recall"] = entry.get("per_defect_recall") or {}
            row["per_defect_support"] = entry.get("per_defect_support") or {}
            row["split_val_balance"] = split.get("val_balance")
            row["split_min_train_goods"] = split.get("min_train_goods")
            rows[key] = row
    df = pd.DataFrame(rows.values())
    if not df.empty:
        df = df.sort_values(["category", "model"]).reset_index(drop=True)
    return df


def combine_runs(*frames: pd.DataFrame) -> pd.DataFrame:
    frames = [f for f in frames if not f.empty]
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    return df.sort_values(["category", "model", "_dir"]).drop_duplicates(["category", "model"], keep="last").sort_values(["category", "model"]).reset_index(drop=True)


def _shape_msg(df: pd.DataFrame, label: str = "models") -> str:
    if df.empty:
        return f"  0  (0 cats x 0 {label})"
    return f"{len(df):3d}  ({df['category'].nunique()} cats x {df['model'].nunique()} {label})"


# Skip val_defect dirs and the trained-model dirs when ingesting feature-model JobA.
df_clean_feature = load_runs(
    JOBA_CLEAN_DIR,
    prefix="jobA_",
    models=FEATURE_MODELS,
    skip_substrings=("val_defect", "_anomalib_", "_rd4ad_"),
)

df_vd_feature = load_runs(
    JOBA_VD_DIR,
    prefix="jobA_val_defect_",
    models=FEATURE_MODELS,
    skip_substrings=("_anomalib_csflow_", "_anomalib_stfpm_", "_anomalib_draem_", "_rd4ad_"),
)

trained_clean_parts = []
trained_vd_parts = []
for adapter in TRAINED_MODELS:
    pf = f"jobA_{adapter}_"
    trained_clean_parts.append(load_runs(JOBA_CLEAN_DIR, prefix=pf, models=[adapter]))
    trained_vd_parts.append(load_runs(JOBA_VD_DIR, prefix=pf, models=[adapter]))

df_clean_trained = combine_runs(*trained_clean_parts)
df_vd_trained = combine_runs(*trained_vd_parts)
df_clean = combine_runs(df_clean_feature, df_clean_trained)
df_vd = combine_runs(df_vd_feature, df_vd_trained)
trained_frames = [
    df_clean_trained.assign(variant="clean"),
    df_vd_trained.assign(variant="val_defect"),
]
trained_frames = [df for df in trained_frames if not df.empty]
df_trained = pd.concat(trained_frames, ignore_index=True) if trained_frames else pd.DataFrame()

loaded_models: set[str] = set()
for df in (df_clean, df_vd):
    if not df.empty:
        loaded_models.update(df["model"].tolist())
MODEL_ORDER = [m for m in ALL_MODELS if m in loaded_models]

print(f"clean feature cells: {_shape_msg(df_clean_feature)}")
print(f"clean trained cells: {_shape_msg(df_clean_trained)}")
print(f"clean all cells:     {_shape_msg(df_clean)}")
print(f"vd feature cells:    {_shape_msg(df_vd_feature)}")
print(f"vd trained cells:    {_shape_msg(df_vd_trained)}")
print(f"vd all cells:        {_shape_msg(df_vd)}")
print(f"trained raw cells:   {_shape_msg(df_trained, label='rows')}")

## Section A - JobA aggregate (val_f1 baseline)

Same outputs as `analyze_runs.py --prefix jobA_val_defect_` plus the legacy `analyze_jobA.py` extras (top-10 AUPR, hardest/easiest, per-defect recall). Run on the unified `df_vd` so feature-based and trained models stay on the same comparison level.

In [ ]:
active = df_vd  # unified val_f1 view (feature-based + trained). Swap to df_clean for the legacy val_quantile baseline.

print(f"Inspecting {len(active)} cells from {active['category'].nunique()} categories.")
active[["category", "model"] + LEGACY_METRICS[:6] + INDUSTRIAL_METRICS].head(12)

In [ ]:
# Per-model aggregates (means + medians).
agg_cols_mean = [
    "auroc", "aupr", "f1", "precision", "recall",
    "recall_at_fpr_1pct", "recall_at_fpr_5pct",
    "macro_recall", "weighted_recall",
    "ms_per_image", "fps", "peak_vram_mb", "fit_seconds",
]
model_n = active.groupby("model").size()
per_model_mean = active.groupby("model")[agg_cols_mean].mean().round(4)
per_model_median = active.groupby("model")[["auroc", "aupr", "f1", "recall_at_fpr_1pct", "macro_recall"]].median().round(4)
per_model_mean.insert(0, "n", model_n)
per_model_median.insert(0, "n", model_n)

print("=== PER-MODEL MEANS ===")
print(per_model_mean.to_string())
print("\n=== PER-MODEL MEDIANS ===")
print(per_model_median.to_string())

In [ ]:
# Per-category AUROC table + winner counts.
pivot_auroc = active.pivot(index="category", columns="model", values="auroc")
pivot_cols = [m for m in MODEL_ORDER if m in pivot_auroc.columns]
pivot_auroc = pivot_auroc.reindex(columns=pivot_cols).round(3)
pivot_auroc["winner"] = pivot_auroc.idxmax(axis=1).str.replace("anomalib_", "", regex=False)
print("=== PER-CATEGORY AUROC ===")
print(pivot_auroc.to_string())

wins = pivot_auroc["winner"].value_counts()
print("\nWIN COUNT (best AUROC per category):")
print(wins.to_string())

In [ ]:
# Top-10 categories by mean AUPR across the currently loaded models.
ranked_aupr = active.groupby("category")["aupr"].mean().sort_values(ascending=False)
print("=== TOP-10 CATEGORIES BY MEAN AUPR ===")
print(ranked_aupr.head(10).round(4).to_string())

ranked_auroc = active.groupby("category")["auroc"].mean().sort_values()
print("\n=== 5 HARDEST CATEGORIES BY MEAN AUROC ===")
print(ranked_auroc.head(5).round(4).to_string())
print("\n=== 5 EASIEST CATEGORIES BY MEAN AUROC ===")
print(ranked_auroc.tail(5).round(4).to_string())

In [ ]:
# Per-defect recall — long-form table (only cells that carry per_defect_recall).
pdef = []
for _, r in active.iterrows():
    per = r["per_defect_recall"] or {}
    sup = r["per_defect_support"] or {}
    for dtype in sorted(per.keys()):
        pdef.append({
            "category": r["category"],
            "model": r["model"].replace("anomalib_", ""),
            "defect_type": dtype,
            "support": sup.get(dtype, 0),
            "recall": per[dtype],
        })
df_perdef = pd.DataFrame(pdef)
print(f"Per-defect rows: {len(df_perdef)}")
df_perdef.head(15)

## Section B - Clean <-> val_defect comparison

Tables A-D from [PLAN job A_analize val_defect.md Section 6](../PLAN%20job%20A_analize%20val_defect.md). Same logic as `compare_clean_vd.py`, just rendered as DataFrames so the cells are inspectable inline. The notebook now keeps feature-based and trained models together; the inner join still limits each comparison row to cells that exist in both clean and val_defect.

In [ ]:
# Join clean <-> val_defect on (category, model). Inner join restricts to cells
# present in both; those are the ones the comparison is meaningful for.
merged = df_clean.merge(df_vd, on=["category", "model"], how="inner",
                        suffixes=("_clean", "_vd"))

for col in ("auroc", "aupr", "f1", "precision", "recall", "threshold_used",
            "recall_at_fpr_1pct", "recall_at_fpr_5pct", "macro_recall"):
    merged[f"d{col}"] = merged[f"{col}_vd"] - merged[f"{col}_clean"]

print(f"Paired (cat, model) cells: {len(merged)}")
if not merged.empty:
    print(merged.groupby("model").size().rename("paired_cells").to_string())
missing_in_clean = sorted(set(zip(df_vd["category"], df_vd["model"])) -
                          set(zip(df_clean["category"], df_clean["model"])))
if missing_in_clean:
    print("WARNING: vd cells without a clean baseline:")
    for x in missing_in_clean:
        print("  ", x)

In [ ]:
# Table A — per (category, model) full deltas.
table_A_cols = [
    "category", "model",
    "auroc_clean", "auroc_vd", "dauroc",
    "aupr_clean", "aupr_vd", "daupr",
    "f1_clean", "f1_vd", "df1",
    "precision_clean", "precision_vd",
    "recall_clean", "recall_vd",
    "threshold_used_clean", "threshold_used_vd",
    "val_recall_vd",
]
table_A = merged[table_A_cols].copy().round(4)
table_A.to_csv(ANALYSIS_DIR / "tableA_per_cat_model.tsv", sep="\t", index=False)
print(f"Table A: {len(table_A)} rows  ->  {ANALYSIS_DIR / 'tableA_per_cat_model.tsv'}")
table_A

In [ ]:
# Table B — per-model headline (means across paired cells).
def _per_model(df: pd.DataFrame, agg="mean") -> pd.DataFrame:
    g = df.groupby("model")
    if agg == "mean":
        return g[["auroc_clean", "auroc_vd", "dauroc",
                  "aupr_clean", "aupr_vd", "daupr",
                  "f1_clean", "f1_vd", "df1",
                  "recall_clean", "recall_vd", "drecall",
                  "precision_clean", "precision_vd"]].mean().round(4)
    return g[["f1_clean", "f1_vd", "recall_clean", "recall_vd"]].median().round(4)

table_B_mean = _per_model(merged, "mean")
table_B_median = _per_model(merged, "median")

table_B = table_B_mean.copy()
for col in ("f1", "recall"):
    for side in ("clean", "vd"):
        table_B[f"{col}_{side}_median"] = table_B_median[f"{col}_{side}"]
table_B.insert(0, "n", merged.groupby("model").size())
table_B.to_csv(ANALYSIS_DIR / "tableB_per_model.tsv", sep="\t")
print(f"Table B  ->  {ANALYSIS_DIR / 'tableB_per_model.tsv'}")
table_B

In [ ]:
# Table C — threshold shift summary.
merged["thr_ratio"] = merged["threshold_used_vd"] / merged["threshold_used_clean"]
table_C = merged.groupby("model")["thr_ratio"].agg(
    n="count", ratio_mean="mean", ratio_median="median",
    ratio_std="std", ratio_min="min", ratio_max="max",
).round(3)
table_C.to_csv(ANALYSIS_DIR / "tableC_threshold_shift.tsv", sep="\t")
print(f"Table C  ->  {ANALYSIS_DIR / 'tableC_threshold_shift.tsv'}")
table_C

In [ ]:
# Table D — val-set sanity (val_defect only).
table_D_cols = ["category", "model", "val_samples",
                "val_precision", "val_recall", "val_f1",
                "val_auroc", "val_aupr", "threshold_used"]
table_D = df_vd[table_D_cols].copy().round(4)
table_D.to_csv(ANALYSIS_DIR / "tableD_val_sanity.tsv", sep="\t", index=False)
print(f"Table D  ->  {ANALYSIS_DIR / 'tableD_val_sanity.tsv'}")
table_D.head(12)

In [ ]:
# Sanity gates from PLAN §8.4.
def _gates(merged: pd.DataFrame, vd: pd.DataFrame) -> None:
    print("=== SANITY GATES ===")
    zero_val = vd[vd["val_recall"].fillna(0) == 0][["category", "model", "val_recall"]]
    print(f"[1] cells with val_recall == 0 : {len(zero_val)}")
    if not zero_val.empty:
        print(zero_val.to_string(index=False))

    da = merged["dauroc"].abs()
    print(f"[2] |dAUROC| median = {da.median():.4f} (target < 0.01)")
    print(f"    |dAUROC| mean   = {da.mean():.4f}")
    print(f"    |dAUROC| max    = {da.max():.4f}")

    print("[3] per-category models with df1 > 0:")
    cat_df1 = merged.groupby("category")["df1"].apply(lambda s: (s > 0).sum())
    cat_n = merged.groupby("category")["df1"].count()
    fails = []
    for cat in sorted(merged["category"].unique()):
        wins = int(cat_df1[cat]); tot = int(cat_n[cat])
        flag = "  <-- FAIL" if wins < 2 else ""
        if wins < 2:
            fails.append(cat)
        print(f"    {cat:20s}  {wins}/{tot}{flag}")
    print(f"    cats failing 2/3 df1>0 : {fails or 'none'}")

    out = merged[merged["dauroc"].abs() > 0.02][["category", "model", "dauroc"]]
    print(f"[4] cells with |dAUROC| > 0.02: {len(out)}")
    if not out.empty:
        print(out.round(4).to_string(index=False))

    crash = merged[merged["precision_vd"] < 0.5][["category", "model", "precision_vd"]]
    print(f"[5] cells with Precision_vd < 0.5: {len(crash)}")
    if not crash.empty:
        print(crash.round(3).to_string(index=False))

    same = merged[(merged["threshold_used_vd"] - merged["threshold_used_clean"]).abs() < 1e-6]
    print(f"[6] cells with threshold_vd == threshold_clean (silent fallback): {len(same)}")
    if not same.empty:
        print(same[["category", "model", "threshold_used_clean", "threshold_used_vd"]].to_string(index=False))

_gates(merged, df_vd)

## §C — Plots

In [ ]:
# F1-lift bar chart, sorted by mean lift across the paired models.
lift = merged.pivot(index="category", columns="model", values="df1")
lift_cols = [m for m in MODEL_ORDER if m in lift.columns]
lift = lift.reindex(columns=lift_cols)
lift["_mean"] = lift.mean(axis=1)
lift = lift.sort_values("_mean")
lift = lift.drop(columns="_mean")

fig, ax = plt.subplots(figsize=(11, 5))
lift.plot(kind="bar", ax=ax, width=0.8)
ax.axhline(0, color="black", linewidth=0.6)
ax.set_ylabel("F1 (val_defect) - F1 (clean)")
ax.set_xlabel("category (sorted by mean lift)")
ax.set_title("F1 lift per (category, model) - clean (val_quantile) -> val_defect (val_f1)")
ax.legend(title="model", labels=[m.replace("anomalib_", "") for m in lift.columns])
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# AUROC stability scatter: clean vs val_defect, identity line = perfect stability.
fig, ax = plt.subplots(figsize=(6, 6))
for m, sub in merged.groupby("model"):
    ax.scatter(sub["auroc_clean"], sub["auroc_vd"],
               label=m.replace("anomalib_", ""), s=40, alpha=0.7)
lims = [0.5, 1.005]
ax.plot(lims, lims, color="gray", linestyle="--", linewidth=1)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("AUROC (clean)")
ax.set_ylabel("AUROC (val_defect)")
ax.set_title("AUROC stability — should hug y=x; outliers are seed/grouping noise")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Threshold-shift histogram per model (val_f1 / val_quantile ratio).
plot_models = [m for m in MODEL_ORDER if m in set(merged["model"])]
if not plot_models:
    raise ValueError("No paired models available for the threshold-shift plot.")

fig, axes = plt.subplots(1, len(plot_models), figsize=(4.2 * len(plot_models), 3.5), sharey=True, squeeze=False)
axes = axes.ravel()
for ax, m in zip(axes, plot_models):
    sub = merged[merged["model"] == m]["thr_ratio"]
    ax.hist(sub.dropna(), bins=12, color="steelblue", edgecolor="black", alpha=0.8)
    ax.axvline(1.0, color="red", linestyle="--", linewidth=1)
    ax.set_title(f"{m.replace('anomalib_', '')}\nmean={sub.mean():.2f}, median={sub.median():.2f}")
    ax.set_xlabel("threshold_vd / threshold_clean")
axes[0].set_ylabel("# categories")
fig.suptitle("Threshold shift (<1 = val_f1 picked a more permissive cut than val_quantile)")
plt.tight_layout()
plt.show()

---

## Pointers

- Source plan: [PLAN job A_analize val_defect.md](../PLAN%20job%20A_analize%20val_defect.md)
- Methodology context: [METHOD.md §2 + §7](../METHOD.md)
- Companion notebook: [compare jobB vs jobA.ipynb](compare%20jobB%20vs%20jobA.ipynb)
- TSV outputs land under `data/outputs/jobA_val_defect_V1/_analysis/`.